In [ ]:
# Start from root
%cd /

# Create and navigate to workspace
!mkdir -p /workspace
%cd /workspace

# Remove old repo if exists
!rm -rf finetune-dk

# Clone fresh
!git clone https://github.com/anderskruse/finetune-dk.git
%cd finetune-dk

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!~/.local/bin/uv venv .venv --clear
!~/.local/bin/uv pip install -e . --python .venv/bin/python
# Required for Qwen3.5 linear attention fast path
!.venv/bin/pip install causal-conv1d flash-linear-attention

In [ ]:
!.venv/bin/python -c "from huggingface_hub import login; login(token='YOUR_HF_TOKEN')"

In [ ]:
!.venv/bin/python -c "from huggingface_hub import whoami; import json; print(json.dumps(whoami(), indent=2))"

In [5]:
!.venv/bin/python -c "import torch; print(f'PyTorch: {torch.__version__}')"

PyTorch: 2.10.0+cu128


In [ ]:
# Pre-finetune test — calls HF Inference API, no GPU needed
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"  # paste your token here
!.venv/bin/python pre_finetune_test.py

In [ ]:
# Clear any lingering GPU processes from previous runs
import subprocess, time

# Show full nvidia-smi to see all processes
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# Kill all compute processes
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader"],
    capture_output=True, text=True
)
pids = [line.split(",")[0].strip() for line in result.stdout.strip().splitlines() if line.strip()]

if pids:
    for pid in pids:
        r = subprocess.run(["kill", "-9", pid], capture_output=True, text=True)
        print(f"Killed PID {pid}: {r.stderr.strip() or 'ok'}")
else:
    print("No compute processes found via nvidia-smi")

time.sleep(3)
print("\nAfter kill:")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True .venv/bin/python train.py --no-thinking --epochs 2

In [ ]:
!.venv/bin/python evaluate.py --model ./outputs/lora_adapter --no-thinking